In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("env", "")
env = dbutils.widgets.get("env")

In [0]:
%run ../utils/common_utils

In [0]:
config = get_config(env = f"{env}", 
                    path = "../utils/config.json")

In [0]:
from pyspark.sql.functions import *

In [0]:
catalog = config.get("catalog")

bronze_schema = config.get("bronze_schema")
silver_schema = config.get("silver_schema")
gold_schema = config.get("gold_schema")

bronze_volume = config.get("bronze_volume")
silver_volume = config.get("silver_volume")
gold_volume = config.get("gold_volume")

In [0]:
class directory_hierarchy:
  def __init__(self, volume_path):
    self.volume_path = volume_path

  def reset_directories(self, layer, directory_list):
    base_dir = f"{self.volume_path}/{layer}"
    dbutils.fs.rm(base_dir, True)
    print(f"Deleted {base_dir}")

    dbutils.fs.mkdirs(base_dir)
    print(f"Created {base_dir}")

    for dir_ in directory_list:
      dbutils.fs.rm(f"{base_dir}/{dir_}", True)
      print(f"Deleted {base_dir}/{dir_}")

      dbutils.fs.mkdirs(f"{base_dir}/{dir_}")
      print(f"Created {base_dir}/{dir_}")

In [0]:
project_dir_tup_ = [
    (bronze_schema, bronze_volume, "landing", ["invoices_data"]),
    (bronze_schema, bronze_volume, "bronze", ["schemas", "checkpoints"]),
    (silver_schema, silver_volume, "silver", ["checkpoints"]),
    (gold_schema, gold_volume, "gold", ["checkpoints"])
]
for schema, volume, dir_, dir_list in project_dir_tup_:
    volume_path = f"/Volumes/{catalog}/{schema}/{volume}"
    dir_process = directory_hierarchy(volume_path)
    dir_process.reset_directories(dir_, dir_list)